# Data Exploration & Visualization

Explore CPAISD / BraTS / RSNA datasets: sample images, mask overlays, class distribution, slice statistics.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0, 'font.size': 11})

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
from configs.config import load_config
from datasets.factory import get_dataset_class

print('Setup complete')

## 1. Load Dataset

In [ ]:
DATASET = "cpaisd"  # Options: cpaisd, cpaisd_enhanced, brats, rsna
SPLIT = "train"      # train / val

config = load_config(DATASET)
dataset_class = get_dataset_class(config.DATASET_NAME)
root_path = config.DATA_PATHS.get(config.DATASET_NAME, config.BASE_PATH)

dataset = dataset_class(
    dataset_root=root_path,
    split=SPLIT,
    T=config.T,
    config=config,
)

print(f"Dataset: {DATASET} ({SPLIT})")
print(f"Samples: {len(dataset)}")
print(f"Image size: {config.IMAGE_SIZE}")
print(f"Num classes: {config.NUM_CLASSES}")

## 2. Class Distribution

In [ ]:
CLASS_NAMES = {
    0: 'Background',
    1: 'Core' if config.NUM_CLASSES >= 3 else 'Abnormality',
    2: 'Penumbra' if config.NUM_CLASSES >= 3 else 'Class 2',
    3: 'Necrosis' if config.NUM_CLASSES >= 4 else 'Class 3',
}

pixel_counts = Counter()
sample_has_class = Counter()

for idx in range(min(len(dataset), 200)):
    _, mask, _ = dataset[idx]
    mask_np = mask.numpy() if torch.is_tensor(mask) else mask
    for c in range(config.NUM_CLASSES):
        count = (mask_np == c).sum()
        pixel_counts[c] += int(count)
        if count > 0:
            sample_has_class[c] += 1

total_pixels = sum(pixel_counts.values())
print(f"Analyzed {min(len(dataset), 200)} samples\n")

print(f"{'Class':<20} {'Pixels':>12} {'%':>8} {'Samples':>8}")
print("-" * 50)
for c in range(config.NUM_CLASSES):
    name = CLASS_NAMES.get(c, f'Class {c}')
    pct = pixel_counts[c] / total_pixels * 100 if total_pixels else 0
    print(f"{name + f' ({c})':<20} {pixel_counts[c]:>12,d} {pct:>7.2f}% {sample_has_class[c]:>8}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

classes = [CLASS_NAMES.get(c, f'Class {c}') for c in range(config.NUM_CLASSES)]
counts = [pixel_counts[c] for c in range(config.NUM_CLASSES)]
colors = ['#555555', '#e74c3c', '#2ecc71', '#3498db', '#f39c12'][:config.NUM_CLASSES]

bars1 = ax1.bar(classes, counts, color=colors, edgecolor='white')
ax1.set_title('Pixel Count per Class', fontweight='bold')
ax1.set_ylabel('Pixel Count')
ax1.tick_params(axis='x', rotation=30)
for bar, count in zip(bars1, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
             f'{count:,}', ha='center', va='bottom', fontsize=9)

sample_counts = [sample_has_class[c] for c in range(config.NUM_CLASSES)]
bars2 = ax2.bar(classes, sample_counts, color=colors, edgecolor='white')
ax2.set_title('Samples Containing Each Class', fontweight='bold')
ax2.set_ylabel('Sample Count')
ax2.tick_params(axis='x', rotation=30)
for bar, count in zip(bars2, sample_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(sample_counts)*0.01,
             f'{count}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to class_distribution.png")

## 3. Sample Visualization

Browse random slices with ground-truth overlay.

In [ ]:
NUM_SAMPLES = 6
idxs = np.random.choice(len(dataset), min(NUM_SAMPLES, len(dataset)), replace=False)

cmap = ListedColormap(['black', '#e74c3c', '#2ecc71', '#3498db', '#f39c12'][:config.NUM_CLASSES])
legend_patches = [
    mpatches.Patch(color=cmap(i), label=CLASS_NAMES.get(i, f'Class {i}'))
    for i in range(1, config.NUM_CLASSES)
]

fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(12, 3 * NUM_SAMPLES))
if NUM_SAMPLES == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(idxs):
    images, mask, metadata = dataset[int(idx)]

    img = images.numpy() if torch.is_tensor(images) else images
    msk = mask.numpy() if torch.is_tensor(mask) else mask

    if img.ndim == 3:
        img_display = img[img.shape[0] // 2]
    elif img.ndim == 2:
        img_display = img
    else:
        img_display = img[0]

    axes[row, 0].imshow(img_display, cmap='gray')
    axes[row, 0].set_title(f'Slice {idx} — CT', fontsize=10, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(msk, cmap=cmap, vmin=0, vmax=config.NUM_CLASSES - 1)
    axes[row, 1].set_title('Ground Truth', fontsize=10, fontweight='bold')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(img_display, cmap='gray')
    axes[row, 2].imshow(msk, cmap=cmap, alpha=0.4, vmin=0, vmax=config.NUM_CLASSES - 1)
    axes[row, 2].set_title('Overlay', fontsize=10, fontweight='bold')
    axes[row, 2].axis('off')

    if isinstance(metadata, dict) and metadata:
        meta_str = ' | '.join(f'{k}: {v}' for k, v in list(metadata.items())[:3])
        fig.text(0.5, 0.98 - row * (0.85 / NUM_SAMPLES), meta_str,
                 ha='center', fontsize=8, style='italic')

fig.legend(handles=legend_patches, loc='lower center', ncol=min(4, config.NUM_CLASSES - 1), fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig('sample_overlays.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to sample_overlays.png")

## 4. HU Window Distribution (CPAISD)

Intensity histogram to verify HU windowing is working correctly.

In [ ]:
pixel_values = []
for idx in range(min(50, len(dataset))):
    images, mask, _ = dataset[idx]
    img = images.numpy() if torch.is_tensor(images) else images
    if img.ndim == 3:
        img = img[img.shape[0] // 2]
    pixel_values.extend(img.flatten().tolist())

pixel_values = np.array(pixel_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(pixel_values, bins=256, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Pixel Intensity Histogram', fontweight='bold')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Frequency')
axes[0].axvline(pixel_values.mean(), color='red', linestyle='--', label=f"Mean: {pixel_values.mean():.3f}")
axes[0].legend()

axes[1].boxplot(pixel_values, vert=False)
axes[1].set_title('Intensity Distribution', fontweight='bold')
axes[1].set_xlabel('Intensity')

plt.tight_layout()
plt.savefig('intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Min: {pixel_values.min():.4f}, Max: {pixel_values.max():.4f}, Mean: {pixel_values.mean():.4f}, Std: {pixel_values.std():.4f}")

## 5. Metadata Stats (CPAISD)

Clinical metadata distribution across samples.

In [ ]:
meta_fields = ['nihss', 'age', 'sex', 'time', 'dsa']
meta_data = {k: [] for k in meta_fields}

for idx in range(min(len(dataset), 500)):
    _, _, metadata = dataset[idx]
    if isinstance(metadata, dict):
        for k in meta_fields:
            if k in metadata:
                v = metadata[k]
                if isinstance(v, torch.Tensor):
                    v = v.item()
                meta_data[k].append(float(v))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

titles = {'nihss': 'NIHSS Score', 'age': 'Age', 'sex': 'Sex (0=M, 1=F)',
          'time': 'Onset Time (hours)', 'dsa': 'DSA (0=No, 1=Yes)'}

for i, k in enumerate(meta_fields):
    vals = np.array(meta_data[k])
    if len(vals) == 0:
        continue
    if k in ('sex', 'dsa'):
        counts = Counter(vals)
        axes[i].bar(counts.keys(), counts.values(), color=['#3498db', '#e74c3c'], edgecolor='white')
        axes[i].set_xticks(list(counts.keys()))
    else:
        axes[i].hist(vals, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].set_title(titles.get(k, k), fontweight='bold')

axes[5].axis('off')
plt.tight_layout()
plt.savefig('metadata_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Metadata stats collected from {len(meta_data['age'])} samples")

## 6. Lesion Size Distribution

How many pixels per class per slice.

In [ ]:
lesion_sizes = {c: [] for c in range(1, config.NUM_CLASSES)}

for idx in range(min(len(dataset), 500)):
    _, mask, _ = dataset[idx]
    msk = mask.numpy() if torch.is_tensor(mask) else mask
    for c in range(1, config.NUM_CLASSES):
        size = (msk == c).sum()
        if size > 0:
            lesion_sizes[c].append(size)

fig, ax = plt.subplots(figsize=(10, 5))
data_to_plot = [lesion_sizes[c] for c in range(1, config.NUM_CLASSES)]
labels = [CLASS_NAMES[c] for c in range(1, config.NUM_CLASSES)]
colors = [cmap(c) for c in range(1, config.NUM_CLASSES)]

bp = ax.boxplot(data_to_plot, patch_artist=True, labels=labels)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

ax.set_title('Lesion Size per Slice (pixels)', fontweight='bold')
ax.set_ylabel('Pixel Count')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('lesion_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()